In [1]:
!pip install sacrebleu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.0/129.0 kB 8.8 MB/s eta 0:00:00


In [2]:

import subprocess, sys, os, random, math, time
import pandas as pd
import sentencepiece as spm
import sacrebleu
import torch, torch.nn as nn, torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import zipfile
from tqdm import tqdm


In [3]:

print(f"PyTorch {torch.__version__} | CUDA {torch.cuda.is_available()}")
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        p = torch.cuda.get_device_properties(i)
        print(f"  GPU {i}: {p.name} — {p.total_memory / 1024**3:.1f} GB")

# ── Paths ──
TRAIN_SRC = "/kaggle/input/datasets/trnthinanh/trungviet/train.zh"
TRAIN_TGT = "/kaggle/input/datasets/trnthinanh/trungviet/train.vi"
TEST_SRC  = "/kaggle/input/datasets/trnthinanh/trungviet/test.zh"
SAVE_DIR  = "./checkpoints"
SPM_ZH_PREFIX = os.path.join(SAVE_DIR, "spm_zh")
SPM_VI_PREFIX = os.path.join(SAVE_DIR, "spm_vi")

# ── Model ──
VOCAB_SIZE     = 12000
HID_SIZE       = 512
FF_SIZE        = 2048
N_HEADS        = 8
N_ENC_LAYERS   = 6
N_DEC_LAYERS   = 6
DROPOUT        = 0.25
DROPPATH_RATE  = 0.10
LABEL_SMOOTHING = 0.1

# ── Training ──
BATCH_SIZE     = 256
ACCUM_STEPS    = 4          # effective batch = 1024
EPOCHS         = 120
LR             = 0.0005
MAX_LEN        = 80
SEED           = 42
SHOW_PROGRESS_BARS = False
PATIENCE       = 20
GRAD_CLIP      = 1.0
BEAM_SIZE      = 16         # ↑8→16: target ~8GB VRAM (3072 seqs × 1.8MB = 5.5GB cache)
BEAM_EVAL_CHUNK = 192       # 192×16=3072 seqs → ~5.5GB cache + ~2.2GB training overhead ≈ 7.7GB
SRC_TOKEN_DROP_PROB = 0.15  # source token dropout augmentation

# ── LR schedule ──
WARMUP_EPOCHS  = 6
MIN_LR_RATIO   = 0.01

# ── Hardware ──
USE_DATA_PARALLEL = True
NUM_GPUS = torch.cuda.device_count()
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE} ({NUM_GPUS} GPU, DataParallel={'ON' if USE_DATA_PARALLEL and NUM_GPUS>1 else 'OFF'})")
torch.backends.cudnn.benchmark = True

os.makedirs(SAVE_DIR, exist_ok=True)
random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)


PyTorch 2.10.0+cu128 | CUDA True
  GPU 0: Tesla T4 — 14.6 GB
  GPU 1: Tesla T4 — 14.6 GB
Device: cuda (2 GPU, DataParallel=ON)


In [4]:

def read_lines(path):
    with open(path, "r", encoding="utf-8") as f:
        return [l.strip() for l in f if l.strip()]

train_src = read_lines(TRAIN_SRC)
train_tgt = read_lines(TRAIN_TGT)
test_src  = read_lines(TEST_SRC)
print(f"Train: {len(train_src)} | Test: {len(test_src)}")


Train: 25648 | Test: 6413


In [5]:

def train_spm(input_file, model_prefix, vocab_size=VOCAB_SIZE, character_coverage=0.9995):
    args_str = (
        f"--input={input_file} --model_prefix={model_prefix} --vocab_size={vocab_size} "
        f"--model_type=bpe --character_coverage={character_coverage} "
        "--pad_id=0 --unk_id=1 --bos_id=2 --eos_id=3 --num_threads=4"
    )
    log_path = model_prefix + "_train.log"
    child_script = "import sentencepiece as spm\n" + f"spm.SentencePieceTrainer.Train({args_str!r})\n"
    t0 = time.time()
    with open(log_path, "w", encoding="utf-8") as lf:
        proc = subprocess.Popen([sys.executable, "-c", child_script], stdout=lf, stderr=subprocess.STDOUT)
        while proc.poll() is None:
            time.sleep(5); print(f"  ... training {os.path.basename(model_prefix)} ({time.time()-t0:.0f}s)", flush=True)
    if proc.returncode != 0:
        raise RuntimeError(f"SPM failed (exit {proc.returncode}). See {log_path}")
    print(f"Trained: {model_prefix}.model ({time.time()-t0:.1f}s)", flush=True)

def load_sp(p):
    sp = spm.SentencePieceProcessor(); sp.Load(p); return sp

tmp_zh, tmp_vi = os.path.join(SAVE_DIR, 'tmp_zh.txt'), os.path.join(SAVE_DIR, 'tmp_vi.txt')
if not os.path.exists(SPM_ZH_PREFIX + '.model'):
    with open(tmp_zh, 'w', encoding='utf-8') as f: f.write('\n'.join(train_src))
    train_spm(tmp_zh, SPM_ZH_PREFIX)
if not os.path.exists(SPM_VI_PREFIX + '.model'):
    with open(tmp_vi, 'w', encoding='utf-8') as f: f.write('\n'.join(train_tgt))
    train_spm(tmp_vi, SPM_VI_PREFIX)

sp_zh = load_sp(SPM_ZH_PREFIX + '.model')
sp_vi = load_sp(SPM_VI_PREFIX + '.model')
print(f"ZH vocab={sp_zh.GetPieceSize()}, VI vocab={sp_vi.GetPieceSize()}")


  ... training spm_zh (5s)
Trained: ./checkpoints/spm_zh.model (5.0s)
  ... training spm_vi (5s)
Trained: ./checkpoints/spm_vi.model (5.0s)
ZH vocab=12000, VI vocab=12000


In [6]:

def tokenize_and_pad(texts, sp, max_len=MAX_LEN):
    all_ids = []
    max_seen = 0
    for t in texts:
        ids = [2] + sp.EncodeAsIds(t)[:max_len - 2] + [3]
        max_seen = max(max_seen, len(ids))
        all_ids.append(ids)
    for ids in all_ids:
        ids.extend([0] * (max_seen - len(ids)))
    return torch.tensor(all_ids, dtype=torch.long)

train_src_ids = tokenize_and_pad(train_src, sp_zh)
train_tgt_ids = tokenize_and_pad(train_tgt, sp_vi)
test_src_ids  = tokenize_and_pad(test_src, sp_zh)
print(f"Train src: {train_src_ids.shape}, tgt: {train_tgt_ids.shape}")
print(f"Test src:  {test_src_ids.shape}")


Train src: torch.Size([25648, 43]), tgt: torch.Size([25648, 75])
Test src:  torch.Size([6413, 41])


In [7]:

class TokenDataset(Dataset):
    def __init__(self, src_ids, tgt_ids):
        self.src, self.tgt = src_ids, tgt_ids
        self.src_lengths = (src_ids != 0).sum(dim=1)
        self.tgt_lengths = (tgt_ids != 0).sum(dim=1)
    def __len__(self): return len(self.src)
    def __getitem__(self, i):
        return self.src[i, :self.src_lengths[i]], self.tgt[i, :self.tgt_lengths[i]]

def collate_fn(batch):
    srcs, tgts = zip(*batch)
    S = max(x.size(0) for x in srcs); T = max(x.size(0) for x in tgts)
    src_pad = torch.zeros(len(batch), S, dtype=torch.long)
    tgt_pad = torch.zeros(len(batch), T, dtype=torch.long)
    for i, (s, t) in enumerate(zip(srcs, tgts)):
        src_pad[i, :s.size(0)] = s; tgt_pad[i, :t.size(0)] = t
    return src_pad, tgt_pad

class BucketBatchSampler(torch.utils.data.Sampler):
    def __init__(self, lengths, batch_size, bucket_mult=50, shuffle=True, seed=0):
        self.lengths, self.bs = lengths, batch_size
        self.bucket_size = batch_size * bucket_mult
        self.shuffle, self.epoch, self.seed = shuffle, 0, seed
    def __iter__(self):
        g = random.Random(self.seed + self.epoch)
        idx = list(range(len(self.lengths)))
        if self.shuffle: g.shuffle(idx)
        batches = []
        for i in range(0, len(idx), self.bucket_size):
            bucket = sorted(idx[i:i+self.bucket_size], key=lambda j: self.lengths[j])
            batches.extend(bucket[j:j+self.bs] for j in range(0, len(bucket), self.bs))
        if self.shuffle: g.shuffle(batches)
        self.epoch += 1; return iter(batches)
    def __len__(self): return (len(self.lengths) + self.bs - 1) // self.bs

# Source-disjoint split
groups = {}
for i, s in enumerate(train_src): groups.setdefault(s, []).append(i)
keys = list(groups); random.Random(SEED).shuffle(keys)
train_idx, valid_idx = [], []
tgt_n = int(0.9 * len(train_src))
for k in keys:
    (train_idx if len(train_idx) < tgt_n else valid_idx).extend(groups[k])

train_ds = TokenDataset(train_src_ids[train_idx], train_tgt_ids[train_idx])
valid_ds = TokenDataset(train_src_ids[valid_idx], train_tgt_ids[valid_idx])
train_lengths = [max(int(s), int(t)) for s, t in zip(train_ds.src_lengths, train_ds.tgt_lengths)]
train_sampler = BucketBatchSampler(train_lengths, BATCH_SIZE, seed=SEED)
dataloader = DataLoader(train_ds, batch_sampler=train_sampler, collate_fn=collate_fn,
                        num_workers=2, pin_memory=True, persistent_workers=True, prefetch_factor=4)
valid_loader = DataLoader(valid_ds, batch_size=256, shuffle=False, collate_fn=collate_fn,
                          num_workers=2, pin_memory=True, persistent_workers=True)
print(f"Train: {len(train_ds)} ({len(dataloader)} batches) | Valid: {len(valid_ds)} ({len(valid_loader)} batches)")


Train: 23083 (91 batches) | Valid: 2565 (11 batches)


In [8]:

# ─────────────────────────────────────────────
#  KV-cache Transformer for fast inference
# ─────────────────────────────────────────────

class DropPath(nn.Module):
    def __init__(self, p=0.0):
        super().__init__(); self.p = p
    def forward(self, x):
        if not self.training or self.p == 0: return x
        keep = 1.0 - self.p
        return x * (keep + torch.rand(x.shape[0], *([1]*(x.ndim-1)), device=x.device)).floor_() / keep


class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000, dropout=0.1):
        super().__init__()
        self.dropout = nn.Dropout(dropout)
        pe = torch.zeros(max_len, d_model)
        pos = torch.arange(max_len, dtype=torch.float).unsqueeze(1)
        div = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(pos * div); pe[:, 1::2] = torch.cos(pos * div)
        self.register_buffer('pe', pe.unsqueeze(0))       # (1, max_len, D)

    def forward(self, x, offset=0):
        """offset: starting position for incremental decoding."""
        return self.dropout(x + self.pe[:, offset:offset + x.size(1)])


def _sdpa(Q, K, V, attn_mask=None, dropout_p=0.0, training=False):
    """Scaled dot-product attention.  Q,K,V: (B, H, T, Dk)"""
    d_k = Q.size(-1)
    scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(d_k)
    if attn_mask is not None:
        scores = scores + attn_mask
    attn = torch.softmax(scores, dim=-1)
    if training and dropout_p > 0:
        attn = F.dropout(attn, p=dropout_p)
    return torch.matmul(attn, V)


# ── Encoder (unchanged from v2) ──
class EncoderLayer(nn.Module):
    def __init__(self, d_model, n_heads, ff_size, dropout=0.1, drop_path=0.0):
        super().__init__()
        self.self_attn = nn.MultiheadAttention(d_model, n_heads, dropout=dropout, batch_first=True)
        self.ff = nn.Sequential(nn.Linear(d_model, ff_size), nn.GELU(), nn.Dropout(dropout),
                                nn.Linear(ff_size, d_model))
        self.norm1 = nn.LayerNorm(d_model); self.norm2 = nn.LayerNorm(d_model)
        self.dropout1 = nn.Dropout(dropout); self.dropout2 = nn.Dropout(dropout)
        self.drop_path = DropPath(drop_path) if drop_path > 0 else nn.Identity()

    def forward(self, src, src_key_padding_mask=None):
        x = self.norm1(src)
        a, _ = self.self_attn(x, x, x, key_padding_mask=src_key_padding_mask, need_weights=False)
        src = src + self.drop_path(self.dropout1(a))
        x = self.norm2(src)
        src = src + self.drop_path(self.dropout2(self.ff(x)))
        return src


# ── Decoder with KV-cache ──
class CachedDecoderLayer(nn.Module):
    """Self-attention uses manual Q/K/V projections for KV-cache.
       Cross-attention uses nn.MHA (recomputed each step; memory is fixed & small)."""
    def __init__(self, d_model, n_heads, ff_size, dropout=0.1, drop_path=0.0):
        super().__init__()
        self.n_heads, self.head_dim = n_heads, d_model // n_heads
        # Self-attention projections
        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        self.W_o = nn.Linear(d_model, d_model)
        # Cross-attention (standard)
        self.cross_attn = nn.MultiheadAttention(d_model, n_heads, dropout=dropout, batch_first=True)
        # FFN
        self.ff = nn.Sequential(nn.Linear(d_model, ff_size), nn.GELU(), nn.Dropout(dropout),
                                nn.Linear(ff_size, d_model))
        # Norms & regularisation
        self.norm1 = nn.LayerNorm(d_model); self.norm2 = nn.LayerNorm(d_model); self.norm3 = nn.LayerNorm(d_model)
        self.drop1 = nn.Dropout(dropout); self.drop2 = nn.Dropout(dropout); self.drop3 = nn.Dropout(dropout)
        self.dp1 = DropPath(drop_path) if drop_path > 0 else nn.Identity()
        self.dp2 = DropPath(drop_path) if drop_path > 0 else nn.Identity()
        self.dp3 = DropPath(drop_path) if drop_path > 0 else nn.Identity()

    def forward(self, tgt, memory, tgt_mask=None, memory_key_padding_mask=None,
                self_attn_cache=None):
        B, T, D = tgt.shape
        H, Hd = self.n_heads, self.head_dim

        # ── Self-attention (with KV-cache) ──
        x = self.norm1(tgt)
        q = self.W_q(x).reshape(B, T, H, Hd).transpose(1, 2)       # (B, H, T, Hd)
        k = self.W_k(x).reshape(B, T, H, Hd).transpose(1, 2)
        v = self.W_v(x).reshape(B, T, H, Hd).transpose(1, 2)
        if self_attn_cache is not None:
            k = torch.cat([self_attn_cache[0], k], dim=2)            # (B, H, T_prev+T, Hd)
            v = torch.cat([self_attn_cache[1], v], dim=2)
        new_self_cache = (k, v)
        # tgt_mask: causal for training (Tq,Tk), None for incremental (Tq=1)
        a = _sdpa(q, k, v, attn_mask=tgt_mask,
                  dropout_p=self.drop1.p if self.training else 0, training=self.training)
        a = a.transpose(1, 2).reshape(B, T, D)
        tgt = tgt + self.dp1(self.drop1(self.W_o(a)))

        # ── Cross-attention (recompute from fixed memory — cheap) ──
        x = self.norm2(tgt)
        a, _ = self.cross_attn(x, memory, memory,
                               key_padding_mask=memory_key_padding_mask, need_weights=False)
        tgt = tgt + self.dp2(self.drop2(a))

        # ── FFN ──
        x = self.norm3(tgt)
        tgt = tgt + self.dp3(self.drop3(self.ff(x)))
        return tgt, new_self_cache


class TransformerSeq2Seq(nn.Module):
    def __init__(self, src_vocab, tgt_vocab, d_model, n_heads, ff_size,
                 n_enc_layers, n_dec_layers, dropout=0.1, drop_path=0.0,
                 pad_idx=0, tie_embeddings=True):
        super().__init__()
        self.d_model, self.pad_idx, self.scale = d_model, pad_idx, math.sqrt(d_model)
        self.src_emb = nn.Embedding(src_vocab, d_model, padding_idx=pad_idx)
        self.tgt_emb = nn.Embedding(tgt_vocab, d_model, padding_idx=pad_idx)
        self.pos_enc = PositionalEncoding(d_model, dropout=dropout)

        enc_dpr = [drop_path * i / max(n_enc_layers-1, 1) for i in range(n_enc_layers)]
        self.encoder_layers = nn.ModuleList(
            [EncoderLayer(d_model, n_heads, ff_size, dropout, enc_dpr[i]) for i in range(n_enc_layers)])
        self.final_norm_enc = nn.LayerNorm(d_model)

        dec_dpr = [drop_path * i / max(n_dec_layers-1, 1) for i in range(n_dec_layers)]
        self.decoder_layers = nn.ModuleList(
            [CachedDecoderLayer(d_model, n_heads, ff_size, dropout, dec_dpr[i]) for i in range(n_dec_layers)])
        self.final_norm_dec = nn.LayerNorm(d_model)

        self.output_proj = nn.Linear(d_model, tgt_vocab)
        if tie_embeddings:
            self.output_proj.weight = self.tgt_emb.weight

    # ── Encode ──
    def encode(self, src):
        mask = (src == self.pad_idx)
        x = self.pos_enc(self.src_emb(src) * self.scale)
        for layer in self.encoder_layers:
            x = layer(x, src_key_padding_mask=mask)
        return self.final_norm_enc(x), mask

    # ── Decode (full, for training) ──
    def decode(self, tgt, memory, memory_mask=None):
        T = tgt.size(1)
        mask = torch.triu(torch.full((T, T), float('-inf'), device=tgt.device), diagonal=1)
        x = self.pos_enc(self.tgt_emb(tgt) * self.scale)
        for layer in self.decoder_layers:
            x, _ = layer(x, memory, tgt_mask=mask, memory_key_padding_mask=memory_mask)
        return self.output_proj(self.final_norm_dec(x))

    # ── Decode step (incremental, for inference) ──
    def decode_step(self, tgt_token, memory, memory_mask, caches, step):
        """Process ONE token.  caches: list of (k,v) tuples or None."""
        x = self.pos_enc(self.tgt_emb(tgt_token) * self.scale, offset=step)
        new_caches = []
        for i, layer in enumerate(self.decoder_layers):
            x, c = layer(x, memory, memory_key_padding_mask=memory_mask,
                         self_attn_cache=caches[i] if caches else None)
            new_caches.append(c)
        return self.output_proj(self.final_norm_dec(x)), new_caches

    # ── Unified forward ──
    def forward(self, src, tgt=None, memory=None, memory_mask=None, mode='train', **kw):
        if mode == 'encode':
            return self.encode(src)
        elif mode == 'decode':
            return self.decode(src, memory, memory_mask)
        elif mode == 'decode_step':
            return self.decode_step(src, memory, memory_mask,
                                    kw.get('caches'), kw.get('step', 0))
        else:                                           # training
            mem, mem_mask = self.encode(src)
            return self.decode(tgt[:, :-1], mem, mem_mask)


model = TransformerSeq2Seq(
    src_vocab=sp_zh.GetPieceSize(), tgt_vocab=sp_vi.GetPieceSize(),
    d_model=HID_SIZE, n_heads=N_HEADS, ff_size=FF_SIZE,
    n_enc_layers=N_ENC_LAYERS, n_dec_layers=N_DEC_LAYERS,
    dropout=DROPOUT, drop_path=DROPPATH_RATE, tie_embeddings=True,
).to(DEVICE)

if USE_DATA_PARALLEL and NUM_GPUS > 1:
    model = nn.DataParallel(model)
    print(f"DataParallel × {NUM_GPUS}")

base = model.module if hasattr(model, 'module') else model
params = sum(p.numel() for p in base.parameters() if p.requires_grad)
with torch.no_grad():
    nn.init.normal_(base.src_emb.weight, std=0.02); nn.init.normal_(base.tgt_emb.weight, std=0.02)
    base.src_emb.weight[0].zero_(); base.tgt_emb.weight[0].zero_()
print(f"Model: {params:,} params | Pre-norm + GELU + DropPath({DROPPATH_RATE}) + KV-cache")


DataParallel × 2
Model: 56,440,544 params | Pre-norm + GELU + DropPath(0.1) + KV-cache


In [9]:

def source_token_dropout(src, pad_idx=0, p=SRC_TOKEN_DROP_PROB):
    """Randomly replace source tokens with PAD (training augmentation)."""
    return src * (torch.rand_like(src.float()) >= p).long()


def train_epoch(model, dataloader, criterion, optimizer, scaler, scheduler=None):
    model.train()
    total_loss = 0
    optimizer.zero_grad(set_to_none=True)
    pbar = tqdm(dataloader, desc='Train', leave=False, disable=not SHOW_PROGRESS_BARS)
    for step, (src, tgt) in enumerate(pbar):
        src = source_token_dropout(src.to(DEVICE, non_blocking=True))
        tgt = tgt.to(DEVICE, non_blocking=True)
        with torch.amp.autocast('cuda'):
            out = model(src, tgt)
            loss = criterion(out.reshape(-1, out.size(-1)), tgt[:, 1:].reshape(-1)) / ACCUM_STEPS
        if torch.isnan(loss) or torch.isinf(loss):
            optimizer.zero_grad(set_to_none=True); continue
        scaler.scale(loss).backward()
        total_loss += loss.item() * ACCUM_STEPS
        if (step + 1) % ACCUM_STEPS == 0 or step + 1 == len(dataloader):
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
            scaler.step(optimizer); scaler.update()
            optimizer.zero_grad(set_to_none=True)
            if scheduler: scheduler.step()
    return total_loss / len(dataloader)


@torch.no_grad()
def evaluate(model, dataloader, sp_tgt):
    """Greedy decode evaluation — uses KV-cache for speed."""
    model.eval()
    enc = model.module if hasattr(model, 'module') else model
    total_loss = 0; hyps, refs = [], []

    for src, tgt in tqdm(dataloader, desc='Eval', leave=False, disable=not SHOW_PROGRESS_BARS):
        src, tgt = src.to(DEVICE, non_blocking=True), tgt.to(DEVICE, non_blocking=True)
        bs = src.size(0)

        # Loss via full forward
        with torch.amp.autocast('cuda'):
            mem, mem_mask = enc.encode(src)
            out = enc.decode(tgt[:, :-1], mem, mem_mask)
            loss = F.cross_entropy(out.reshape(-1, out.size(-1)), tgt[:, 1:].reshape(-1),
                                   ignore_index=0, label_smoothing=LABEL_SMOOTHING)
        total_loss += loss.item()

        # ── Greedy decode WITH KV-cache ──
        with torch.amp.autocast('cuda', dtype=torch.float16):
            caches = [None] * len(enc.decoder_layers)
            dec_in = torch.full((bs, 1), 2, dtype=torch.long, device=DEVICE)  # BOS
            all_tok = torch.full((bs, MAX_LEN), 0, dtype=torch.long, device=DEVICE)
            finished = torch.zeros(bs, dtype=torch.bool, device=DEVICE)
            for t in range(MAX_LEN):
                logits, caches = enc.decode_step(dec_in, mem, mem_mask, caches, step=t)
                nxt = logits[:, -1].argmax(1)
                all_tok[:, t] = nxt
                finished |= nxt.eq(3)
                if finished.all(): break
                nxt = torch.where(finished, torch.zeros_like(nxt), nxt)
                dec_in = nxt.unsqueeze(1)

        for i in range(bs):
            ids = all_tok[i].tolist()
            if 3 in ids: ids = ids[:ids.index(3)]
            hyps.append(sp_tgt.DecodeIds(ids).replace('_', ' ').strip())
            ref = tgt[i].tolist()[1:]
            if 3 in ref: ref = ref[:ref.index(3)]
            refs.append(sp_tgt.DecodeIds([x for x in ref if x not in (0, 1)]).replace('_', ' ').strip())

    bleu = sacrebleu.corpus_bleu(hyps, [refs], force=True).score
    return total_loss / len(dataloader), bleu


@torch.inference_mode()
def validation_beam_bleu(model, dataloader, sp_tgt, beam_size=BEAM_SIZE, max_len=MAX_LEN,
                         chunk_size=BEAM_EVAL_CHUNK):
    """Beam-search BLEU — uses KV-cache, chunked to avoid OOM."""
    enc = model.module if hasattr(model, 'module') else model
    enc.eval()
    hypotheses, references = [], []

    for src, tgt in tqdm(dataloader, desc='Beam', leave=False, disable=not SHOW_PROGRESS_BARS):
        src = src.to(DEVICE, non_blocking=True)
        tgt = tgt.to(DEVICE, non_blocking=True)
        bs = src.size(0)

        # Process in small chunks to avoid OOM
        for ci in range(0, bs, chunk_size):
            cs = src[ci:ci+chunk_size]
            ct = tgt[ci:ci+chunk_size]
            cb = cs.size(0)
            torch.cuda.empty_cache()

            mem, mem_mask = enc.encode(cs)

            # Expand for beam
            mem = mem.unsqueeze(1).expand(-1, beam_size, -1, -1).reshape(cb*beam_size, mem.size(1), -1)
            mem_mask = mem_mask.unsqueeze(1).expand(-1, beam_size, -1).reshape(cb*beam_size, -1)
            dec_in = torch.full((cb*beam_size, 1), 2, dtype=torch.long, device=DEVICE)
            seqs = dec_in.clone()                    # accumulate generated tokens
            scores = torch.full((cb, beam_size), -float('inf'), device=DEVICE)
            scores[:, 0] = 0; scores = scores.flatten()
            finished = torch.zeros(cb*beam_size, dtype=torch.bool, device=DEVICE)
            offsets = (torch.arange(cb, device=DEVICE)*beam_size).unsqueeze(1)

            caches = [None] * len(enc.decoder_layers)

            for t in range(max_len):
                logits, caches = enc.decode_step(dec_in, mem, mem_mask, caches, step=t)
                log_p = torch.log_softmax(logits[:, -1].float(), dim=-1)
                log_p[finished] = -float('inf'); log_p[finished, 0] = 0
                V = log_p.size(-1)
                cand = (scores.unsqueeze(1) + log_p).view(cb, beam_size * V)
                top_s, top_i = cand.topk(beam_size, dim=-1)
                parent, nxt = top_i // V, top_i % V
                sel = (parent + offsets).reshape(-1)

                dec_in = nxt.reshape(-1, 1)  # only feed NEW token (KV-cache has history)
                seqs = torch.cat([seqs[sel], nxt.reshape(-1, 1)], dim=1)  # track full seq
                scores = top_s.reshape(-1)
                finished = finished[sel] | nxt.reshape(-1).eq(3)

                new_caches = []
                for (ck, cv) in caches:
                    new_caches.append((ck[sel], cv[sel]))
                caches = new_caches

                if finished.all(): break

            lengths = seqs.ne(0).sum(1).float().clamp_min(1)
            norm = (scores / (((5+lengths)/6)**0.6)).view(cb, beam_size)
            best = norm.argmax(-1)
            seqs_out = seqs.view(cb, beam_size, -1)[torch.arange(cb, device=DEVICE), best].cpu().tolist()

            for pred, ref in zip(seqs_out, ct.tolist()):
                pred = pred[1:]
                if 3 in pred: pred = pred[:pred.index(3)]
                ref = ref[1:]
                if 3 in ref: ref = ref[:ref.index(3)]
                hypotheses.append(sp_tgt.DecodeIds([x for x in pred if x not in (0,1)]).replace('_',' ').strip())
                references.append(sp_tgt.DecodeIds([x for x in ref if x not in (0,1)]).replace('_',' ').strip())

    return sacrebleu.corpus_bleu(hypotheses, [references], force=True).score


In [10]:

class CosineScheduleWithWarmup:
    def __init__(self, optimizer, warmup_steps, total_steps, min_lr_ratio=MIN_LR_RATIO):
        self.opt = optimizer; self.warmup = warmup_steps; self.total = total_steps
        self.ratio = min_lr_ratio; self.n = 0; self.base = optimizer.param_groups[0]['lr']
    def step(self):
        self.n += 1
        if self.n <= self.warmup:
            lr = self.base * self.n / self.warmup
        else:
            p = (self.n - self.warmup) / max(1, self.total - self.warmup)
            lr = self.base * (self.ratio + 0.5*(1-self.ratio)*(1+math.cos(math.pi*min(p,1))))
        for pg in self.opt.param_groups: pg['lr'] = lr
        return lr
    def get_lr(self): return self.opt.param_groups[0]['lr']


criterion = nn.CrossEntropyLoss(ignore_index=0, label_smoothing=LABEL_SMOOTHING)
optimizer = optim.AdamW(model.parameters(), lr=LR, betas=(0.9, 0.98), eps=1e-9, weight_decay=0.01)
scaler = torch.amp.GradScaler('cuda')

n_opt = max(1, len(dataloader) // ACCUM_STEPS)
warmup_steps = max(100, WARMUP_EPOCHS * n_opt)
total_steps = EPOCHS * n_opt
scheduler = CosineScheduleWithWarmup(optimizer, warmup_steps, total_steps)
print(f"Warmup {warmup_steps} steps, Total {total_steps} steps, {n_opt} opt-steps/epoch")

BEAM_EVAL_EVERY = 8          # ↓ was 4: less frequent beam eval = faster training
best_bleu = 0.0; best_val_loss = float('inf')
best_model_path = os.path.join(SAVE_DIR, 'best_model.pt')
patience_counter = 0; BLEU_MIN_DELTA = 0.05

print("Starting training...\n")
history = []
for epoch in range(1, EPOCHS + 1):
    train_loss = train_epoch(model, dataloader, criterion, optimizer, scaler, scheduler)
    val_loss, greedy_bleu = evaluate(model, valid_loader, sp_vi)
    lr = optimizer.param_groups[0]['lr']

    run_beam = (epoch == 1 or epoch % BEAM_EVAL_EVERY == 0)
    if run_beam:
        torch.cuda.empty_cache()
        with torch.amp.autocast('cuda', dtype=torch.float16):
            beam_bleu = validation_beam_bleu(model, valid_loader, sp_vi)
        if beam_bleu > best_bleu + BLEU_MIN_DELTA:
            best_bleu, best_val_loss, patience_counter = beam_bleu, val_loss, 0
            m = model.module if hasattr(model, 'module') else model
            torch.save({'epoch': epoch, 'model_state_dict': m.state_dict(),
                        'optimizer_state_dict': optimizer.state_dict(), 'bleu': beam_bleu,
                        'greedy_bleu': greedy_bleu, 'train_loss': train_loss,
                        'val_loss': val_loss}, best_model_path)
            marker = ' *** BEST BEAM BLEU ***'
        else:
            patience_counter += BEAM_EVAL_EVERY; marker = ''
        metric = f"greedy={greedy_bleu:.2f} | beam={beam_bleu:.2f}"
    else:
        metric, marker = f"greedy={greedy_bleu:.2f}", ''

    history.append({"epoch": epoch, "train_loss": train_loss, "val_loss": val_loss,
                    "greedy_bleu": greedy_bleu,
                    "beam_bleu": beam_bleu if run_beam else None, "lr": lr})
    pd.DataFrame(history).to_csv(os.path.join(SAVE_DIR, "training_history.csv"), index=False)
    print(f"Epoch {epoch:02d} | Train={train_loss:.3f} | Val={val_loss:.3f} | {metric} | lr={lr:.6f}{marker}")
    if patience_counter >= PATIENCE:
        print(f"\nEarly stopping at epoch {epoch}."); break

print(f"\nBest BLEU: {best_bleu:.2f} | Best val loss: {best_val_loss:.3f}")


Warmup 132 steps, Total 2640 steps, 22 opt-steps/epoch
Starting training...

Epoch 01 | Train=8.017 | Val=7.503 | greedy=0.00 | beam=0.00 | lr=0.000087
Epoch 02 | Train=6.826 | Val=6.166 | greedy=0.00 | lr=0.000174
Epoch 03 | Train=5.832 | Val=5.747 | greedy=0.08 | lr=0.000261
Epoch 04 | Train=5.459 | Val=5.325 | greedy=0.81 | lr=0.000348
Epoch 05 | Train=5.146 | Val=5.129 | greedy=0.30 | lr=0.000436
Epoch 06 | Train=4.857 | Val=4.877 | greedy=1.77 | lr=0.000500
Epoch 07 | Train=4.580 | Val=4.579 | greedy=3.94 | lr=0.000500
Epoch 08 | Train=4.310 | Val=4.295 | greedy=5.97 | beam=5.26 | lr=0.000499 *** BEST BEAM BLEU ***
Epoch 09 | Train=4.060 | Val=4.075 | greedy=7.02 | lr=0.000499
Epoch 10 | Train=3.838 | Val=3.896 | greedy=9.84 | lr=0.000498
Epoch 11 | Train=3.674 | Val=3.770 | greedy=11.26 | lr=0.000497
Epoch 12 | Train=3.521 | Val=3.669 | greedy=13.05 | lr=0.000496
Epoch 13 | Train=3.383 | Val=3.578 | greedy=14.36 | lr=0.000495
Epoch 14 | Train=3.268 | Val=3.571 | greedy=14.46 | lr

In [11]:
print('Double-descent sweep disabled. Set RUN_DOUBLE_DESCENT_SWEEP=True to enable.')

Double-descent sweep disabled. Set RUN_DOUBLE_DESCENT_SWEEP=True to enable.


In [12]:

INFER_BEAM_SIZE  = 16       # ↑8→16: 256×16=4096 seqs × 1.8MB = 7.4GB → ~8GB total
INFER_MAX_LEN    = 100      # allow longer translations
INFER_BATCH_SIZE = 256      # ↑128→256: translate_test chunks to 256 for beam

PAD_ID, BOS_ID, EOS_ID = 0, 2, 3

# Translation memory: exact source overlap train→test
from collections import Counter, defaultdict
_tgt_by_src = defaultdict(list)
for _s, _t in zip(train_src, train_tgt): _tgt_by_src[_s].append(_t)
EXACT_TM = {s: Counter(ts).most_common(1)[0][0] for s, ts in _tgt_by_src.items()}

def _trim(ids):
    last = int((ids != PAD_ID).any(dim=0).nonzero()[-1]) + 1
    return ids[:, :last]

@torch.inference_mode()
def greedy_decode_fast(model, src, max_len=INFER_MAX_LEN):
    """Greedy decode WITH KV-cache — ~20x faster than baseline."""
    enc = model.module if hasattr(model, 'module') else model
    bs = src.size(0)
    mem, mem_mask = enc.encode(src)
    caches = [None] * len(enc.decoder_layers)
    dec_in = torch.full((bs, 1), BOS_ID, dtype=torch.long, device=src.device)
    tokens = torch.full((bs, max_len), PAD_ID, dtype=torch.long, device=src.device)
    finished = torch.zeros(bs, dtype=torch.bool, device=src.device)
    for t in range(max_len):
        logits, caches = enc.decode_step(dec_in, mem, mem_mask, caches, step=t)
        nxt = logits[:, -1].argmax(-1)
        tokens[:, t] = nxt
        finished |= nxt.eq(EOS_ID)
        if finished.all(): break
        nxt = torch.where(finished, torch.zeros_like(nxt), nxt)
        dec_in = nxt.unsqueeze(1)
    return tokens.cpu().tolist()


@torch.inference_mode()
def beam_search_decode_fast(model, src, beam_size=INFER_BEAM_SIZE,
                            max_len=INFER_MAX_LEN, length_penalty=0.6):
    """Beam search WITH KV-cache — massive speedup over baseline."""
    enc = model.module if hasattr(model, 'module') else model
    bs = src.size(0)
    mem, mem_mask = enc.encode(src)
    mem = mem.unsqueeze(1).expand(-1, beam_size, -1, -1).reshape(bs*beam_size, mem.size(1), -1)
    mem_mask = mem_mask.unsqueeze(1).expand(-1, beam_size, -1).reshape(bs*beam_size, -1)
    dec_in = torch.full((bs*beam_size, 1), BOS_ID, dtype=torch.long, device=src.device)
    seqs = dec_in.clone()                                              # track full sequence
    beam_sc = torch.full((bs, beam_size), -float('inf'), device=src.device)
    beam_sc[:, 0] = 0; beam_sc = beam_sc.flatten()
    finished = torch.zeros(bs*beam_size, dtype=torch.bool, device=src.device)
    offsets = (torch.arange(bs, device=src.device)*beam_size).unsqueeze(1)

    caches = [None] * len(enc.decoder_layers)

    for _ in range(max_len):
        logits, caches = enc.decode_step(dec_in, mem, mem_mask, caches, step=_)
        log_p = torch.log_softmax(logits[:, -1].float(), dim=-1)
        log_p[finished] = -float('inf'); log_p[finished, PAD_ID] = 0
        V = log_p.size(-1)
        cand = (beam_sc.unsqueeze(1) + log_p).view(bs, beam_size * V)
        top_s, top_i = cand.topk(beam_size, dim=-1)
        parent, nxt = top_i // V, top_i % V
        sel = (parent + offsets).reshape(-1)
        dec_in = nxt.reshape(-1, 1)                                     # only NEW token for KV-cache
        seqs = torch.cat([seqs[sel], nxt.reshape(-1, 1)], dim=1)       # track full seq
        beam_sc = top_s.reshape(-1)
        finished = finished[sel] | nxt.reshape(-1).eq(EOS_ID)
        caches = [(ck[sel], cv[sel]) for ck, cv in caches]
        if finished.all(): break

    lengths = seqs.ne(PAD_ID).sum(1).float().clamp_min(1)
    sc = (beam_sc / (((5+lengths)/6)**length_penalty)).view(bs, beam_size)
    best = sc.argmax(-1)
    return seqs.view(bs, beam_size, -1)[torch.arange(bs, device=src.device), best].cpu().tolist()


def _ids2text(ids, sp):
    ids = [x for x in ids if x != BOS_ID]
    if EOS_ID in ids: ids = ids[:ids.index(EOS_ID)]
    return sp.DecodeIds(ids).replace('_', ' ').strip()

@torch.inference_mode()
def translate_test(model, test_ids, source_texts, sp_tgt, out_path,
                   bs=INFER_BATCH_SIZE, beam_size=INFER_BEAM_SIZE, max_len=INFER_MAX_LEN):
    model.eval()
    outputs = []
    chunk = min(256, bs) if beam_size > 1 else bs
    for start in tqdm(range(0, len(test_ids), bs), desc='Translate', disable=not SHOW_PROGRESS_BARS):
        batch_src = _trim(test_ids[start:start+bs]).to(DEVICE, non_blocking=True)
        bs_actual = batch_src.size(0)
        with torch.autocast(device_type='cuda', dtype=torch.float16, enabled=DEVICE.type=='cuda'):
            if beam_size == 1:
                ids_batch = greedy_decode_fast(model, batch_src, max_len)
                outputs.extend(_ids2text(ids, sp_tgt) for ids in ids_batch)
            else:
                torch.cuda.empty_cache()
                for ci in range(0, bs_actual, chunk):
                    cs = batch_src[ci:ci+chunk]
                    beam_out = beam_search_decode_fast(model, cs, beam_size, max_len)
                    outputs.extend(_ids2text(ids, sp_tgt) for ids in beam_out)
                torch.cuda.empty_cache()

    exact = 0
    for i, s in enumerate(source_texts):
        if s in EXACT_TM: outputs[i] = EXACT_TM[s]; exact += 1
    print(f"Translation-memory hits: {exact}/{len(source_texts)}")
    assert len(outputs) == len(source_texts)
    pd.DataFrame({'tieng_trung': source_texts, 'tieng_viet': outputs}).to_csv(
        out_path, index=False, encoding='utf-8-sig')
    return out_path


In [13]:

submission_path = "submission.csv"
translate_test(model, test_src_ids, test_src, sp_vi, submission_path,
               bs=INFER_BATCH_SIZE, beam_size=INFER_BEAM_SIZE, max_len=INFER_MAX_LEN)

df = pd.read_csv(submission_path)
print("Sample translations:"); print(df.head(10).to_string())

zip_path = "submission.zip"
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as z:
    z.write(submission_path, arcname="submission.csv")
print(f"\nSubmission: {zip_path} ({os.path.getsize(zip_path)/1024:.1f} KB)")


Translation-memory hits: 497/6413
Sample translations:
               tieng_trung                                     tieng_viet
0      还 早 呀 ， 过 几 年 再 说 。               còn sớm mà , qua mấy năm nữa nói
1                 你 跟 我 去吧                            bạn cùng tôi đi nhé
2                 我 都 喜欢 。                                tôi đều thích .
3      我 受伤 了 需要 一 辆 救护车 。              Tôi tự làm bị một xe cứu thương .
4          等 一下 。 这个 是 吗 ？                    đợi 1 chút này phải không ?
5          我 想 烫 一下 这 套 衣服                     tôi muốn ủi bộ quần áo này
6         配 这个 菜 什么 比较 好 ？                              Thức ăn thế nào ?
7                电影院 在 哪 ？                         Rạp chiếu phim ở đâu ?
8             你 今天 几 点 来 ?                      Bạn hôm_nay mấy giờ đến ?
9  不 , 我 会 在 我 逗留 期间 买 的 。  Không , tôi sẽ mua nó trong nhiều thời gian .

Submission: submission.zip (196.9 KB)
